
# Two-zone RO settlement analysis

This notebook reads the A/B-only RO settlement outputs from `RO_Settlement/Results` and presents the settlement gap together with the congestion-rent recovery check.


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

plt.style.use('default')
plt.rcParams.update({
    'figure.dpi': 140,
    'savefig.dpi': 300,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.linestyle': '--',
    'grid.alpha': 0.25,
})

BASE_DIR = Path(r'C:/Users/Administrator/OneDrive/Thesis Code/cross_border-CM-2zone')
RESULTS_DIR = BASE_DIR / 'RO_Settlement' / 'Results'
FIGURES_DIR = RESULTS_DIR / 'Figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
SAVE_FIGURES = True

def save_fig(fig, name):
    path = FIGURES_DIR / name
    if SAVE_FIGURES:
        fig.savefig(path, bbox_inches='tight')
        print(f'Saved figure: {path}')

def first_existing(df, *cols):
    for col in cols:
        if col in df.columns:
            return col
    raise KeyError(f'None of these columns exist: {cols}')

summary = pd.read_csv(RESULTS_DIR / 'ro_settlement_summary_K500.csv', sep=';')
events = pd.read_csv(RESULTS_DIR / 'ro_settlement_events_K500.csv', sep=';')

contract_col = first_existing(summary, 'Contract')
shortfall_col = first_existing(summary, 'Annual_HedgeShortfall_MEUR')
cr_corr_col = first_existing(summary, 'Annual_CR_Corridor_MEUR')
cr_alloc_col = first_existing(summary, 'Annual_CR_Alloc_MEUR')
omega_col = first_existing(summary, 'Annual_CR_Recovery_Omega_MEUR', 'Annual_FTRlike_Omega_MEUR')
outcome_col = first_existing(summary, 'CR_Recovery_Outcome', 'FTRlike_Outcome')

display(summary[[contract_col, 'Exporter', 'Importer', 'Capacity_MW', shortfall_col, cr_corr_col, cr_alloc_col, omega_col, outcome_col]])


In [ ]:

# Figure 1: annual RO shortfall and congestion-rent recovery
row = summary.iloc[0]
values = pd.Series({
    'Hedge shortfall': row[shortfall_col],
    'Congestion rent': row[cr_corr_col],
    'Recovery allocation': row[cr_alloc_col],
    'Recovery balance': row[omega_col],
})

fig, ax = plt.subplots(figsize=(7.0, 4.5))
colors = ['#D55E00', '#4C72B0', '#55A868', '#8172B2']
bars = ax.bar(values.index, values.values, color=colors, width=0.6)
ax.axhline(0, color='black', linewidth=1.0)
ax.set_ylabel('Annual value [M?/yr]')
ax.set_title('Two-zone RO settlement at K = 500 ?/MWh')
ax.tick_params(axis='x', rotation=15)
for bar, val in zip(bars, values.values):
    ax.annotate(f'{val:,.3f}', (bar.get_x() + bar.get_width() / 2, val),
                xytext=(0, 4 if val >= 0 else -12), textcoords='offset points',
                ha='center', va='bottom' if val >= 0 else 'top', fontsize=9)
save_fig(fig, 'ro_annual_balance_2zone_K500.png')
plt.show()


In [ ]:

# Figure 2: timestep-level shortfall and recovery balance
evt = events.copy()
evt['Timestep'] = evt['Timestep'].astype(int)
evt = evt.sort_values('Timestep')

fig, ax = plt.subplots(figsize=(8.5, 4.6))
ax.axhline(0, color='black', linewidth=1.0)
ax.plot(evt['Timestep'], evt['HedgeShortfall_EUR_per_h'] / 1e6, marker='o', color='#D55E00', label='RO shortfall')
ax.plot(evt['Timestep'], evt['CR_Recovery_Omega_EUR_per_h'] / 1e6, marker='s', color='#55A868', label='Congestion-rent recovery balance')
ax.set_xlabel('Representative timestep')
ax.set_ylabel('Hourly value [M?/h]')
ax.set_title('Event-level settlement mechanism')
ax.legend(frameon=False)
save_fig(fig, 'ro_event_level_2zone_K500.png')
plt.show()


In [ ]:

# Figure 3: weighted scarcity category hours
cat_labels = {
    1: 'Cat 1: importer only above strike',
    2: 'Cat 2: both zones above strike',
    3: 'Cat 3: exporter only above strike',
}
cat_hours = (
    events.groupby('Category')['Weight_h'].sum()
    .reindex([1, 2, 3])
    .fillna(0.0)
    .rename(index=cat_labels)
)

fig, ax = plt.subplots(figsize=(8.0, 4.0))
bars = ax.bar(cat_hours.index, cat_hours.values, color=['#4C72B0', '#55A868', '#C44E52'])
ax.set_ylabel('Weighted hours [h/yr]')
ax.set_title('Scarcity classification in the two-zone run')
ax.tick_params(axis='x', rotation=18)
for bar, val in zip(bars, cat_hours.values):
    ax.annotate(f'{val:,.0f}', (bar.get_x() + bar.get_width() / 2, val),
                xytext=(0, 4), textcoords='offset points', ha='center', va='bottom', fontsize=9)
save_fig(fig, 'ro_scarcity_categories_2zone_K500.png')
plt.show()



### Reading the figures

The first figure compares the annual RO shortfall with the congestion-rent recovery check. The second figure shows how the settlement gap behaves across the weighted scarcity timesteps. The third figure shows how much of the year falls into each scarcity category.
